# Evex Initial Data Analysis

In [ ]:
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, timezone, timedelta, time
from IPython.display import display, Markdown
import pytz
import locale
import plotly as plt
import numpy as np
import json
import plotly.graph_objects as go
import os
from plotly.subplots import make_subplots
from jira import JIRA
import plotly.express as px
import pickle
from jira_loader import fetch_jira_issues
from data_loading import save_data, load_data
from data_transformation import load_issues_Amparex, load_issues

%load_ext autoreload
%autoreload 2
locale.setlocale(locale.LC_TIME, "de_DE.UTF-8")

In [ ]:
load_dotenv(override=True)
JIRA_URL = os.getenv("JIRA_URL")
JIRA_USERNAME = os.getenv("JIRA_USERNAME")
JIRA_PASSWORD = os.getenv("JIRA_PASSWORD")
# Optional global filters in sidebar
start_date = datetime.now(timezone.utc) - timedelta(days=7)
end_date = datetime.now(timezone.utc)
tz = pytz.UTC
start_dt = tz.localize(datetime.combine(start_date, time.min))
end_dt = tz.localize(datetime.combine(end_date, time.max))
jira = JIRA(server=JIRA_URL, basic_auth=(JIRA_USERNAME, JIRA_PASSWORD))
issues = fetch_jira_issues(start_dt, end_dt, project="SDIPR", max_issues=200)

In [ ]:
fields = jira.fields()
for f in fields:
    if "Zentrale" in f["name"] or "Filiale" in f["name"]:
        print(f)

In [ ]:
df = load_issues(issues)

In [ ]:
df

In [ ]:
df.head()

In [ ]:
issue = issues[0]
issue["fields"]["customfield_10010"]["_links"]["agent"]

In [ ]:
dates = [issue["fields"]["created"] for issue in issues]
d = [datetime.strptime(date, "%Y-%m-%dT%H:%M:%S.%f%z") for date in dates]
# fix offset
today = datetime.now(timezone.utc)
# today mines five days
d1 = today - timedelta(days=5)
d2 = today - timedelta(days=2)

d_in = [date for date in d if date >= d1 and date <= d2]
tickets_before = [date for date in d if date < d1]
len(tickets_before)

In [ ]:
start_date = datetime.now(timezone.utc) - timedelta(days=30)
end_date = datetime.now(timezone.utc)
tz = pytz.UTC
start_dt = tz.localize(datetime.combine(start_date, time.min))
end_dt = tz.localize(datetime.combine(end_date, time.max))

In [ ]:
issues = fetch_jira_issues(jira, jql, start_dt, end_dt, max_issues=200)

In [ ]:
max(d)

In [ ]:
today

## Pull Issue list

In [ ]:
# Create a Jira client and authenticate with API key
jira = JIRA(
    server="https://amparex.atlassian.net/",
    basic_auth=("bl@flex.capital", "{{JIRA_API_TOKEN}}"),
)

In [ ]:
run_jira_pull = True
if run_jira_pull:
    jql = "project = SDIPR ORDER BY created DESC"
    issues = fetch_jira_issues(jira, jql, max_issues=1000)
    # dump issues into pickle file
    with open("issues.pkl", "wb") as f:
        pickle.dump(issues, f)

else:
    # retrieve issues from pickle file
    with open("issues.pkl", "rb") as f:
        issues = pickle.load(f)

In [ ]:
len(issues)

In [ ]:
issues[0]["fields"]

In [ ]:
# read json from data/jira-servicedesk-schema-objects.json
with open("data/jira-servicedesk-schema-objects.json", "r") as f:
    schema = json.load(f)

object_id_to_name = {v["id"]: v["name"] for v in schema["values"]}

In [ ]:
df = {
    "key": [],
    "summary": [],
    "description": [],
    "status": [],
    "status_category": [],
    "created": [],
    "updated": [],
    "labels": [],
    "source": [],
    "priority": [],
    "category": [],
    "issuetype": [],
    "main_category_id": [],
    "sub_category_id": [],
    "currentstatus_name": [],
    "currentstatus_date": [],
    "comments": [],
    "request_type": [],
    "priority": [],
}
for issue in issues:
    df["key"].append(issue["key"])
    df["summary"].append(issue["fields"]["summary"])
    df["description"].append(issue["fields"]["description"])
    df["status"].append(issue["fields"]["status"]["name"])
    df["status_category"].append(issue["fields"]["status"]["statusCategory"]["name"])
    # df['creator'].append(issue['fields']['creator']['displayName'])
    df["issuetype"].append(issue["fields"]["issuetype"]["name"])
    df["created"].append(issue["fields"]["created"])
    df["updated"].append(issue["fields"]["updated"])
    df["labels"].append(issue["fields"]["labels"])
    df["priority"].append(issue["fields"]["priority"]["name"])
    df["category"].append(issue["fields"]["customfield_10065"])

    if issue["fields"]["customfield_10010"] is not None:
        df["request_type"].append(
            issue["fields"]["customfield_10010"]["requestType"]["name"]
        )
    else:
        df["request_type"].append("")
    if issue["fields"]["comment"] is not None:
        df["comments"].append(
            "\n\n".join([c["body"] for c in issue["fields"]["comment"]["comments"]])
        )
    else:
        df["comments"].append([])

    try:
        df["currentstatus_name"].append(
            issue["fields"]["customfield_10010"]["currentStatus"]["status"]
        )
        df["currentstatus_date"].append(
            issue["fields"]["customfield_10010"]["currentStatus"]["statusDate"]["jira"]
        )
    except:
        df["currentstatus_name"].append("")
        df["currentstatus_date"].append("")

    try:
        v = issue["fields"]["customfield_10675"]["value"]
        df["source"].append(v)
    except:
        df["source"].append("")

    cf = issue["fields"]["customfield_10680"]
    if len(cf) > 0:
        df["main_category_id"].append(cf[0]["objectId"])
    else:
        df["main_category_id"].append("")
    cf = issue["fields"]["customfield_10679"]
    if len(cf) > 0:
        df["sub_category_id"].append(cf[0]["objectId"])
    else:
        df["sub_category_id"].append("")


df = pd.DataFrame(df)
### convert created, updated to datetime
df["created"] = pd.to_datetime(df["created"], errors="coerce", utc=True)
df["updated"] = pd.to_datetime(df["updated"], errors="coerce", utc=True)
df["currentstatus_date"] = pd.to_datetime(
    df["currentstatus_date"], errors="coerce", utc=True
)
df["time_to_resolution_h"] = (
    df["currentstatus_date"] - df["created"]
).dt.total_seconds() / 3600
df["time_to_resolution_days"] = (df["currentstatus_date"] - df["created"]).dt.days
df["resolution"] = "> 1 day"
df["resolution"] = np.where(df["time_to_resolution_days"] <= 1, "Same day", "> 1 day")
df["bdays"] = np.busday_count(
    df["created"].to_numpy(dtype="datetime64[D]"),
    df["updated"].to_numpy(dtype="datetime64[D]"),
)
df["created_string"] = df["created"].dt.strftime("%Y-%m-%d")
df["updated_string"] = df["updated"].dt.strftime("%Y-%m-%d")
df["year"] = df["created"].dt.year
df["month"] = df["created"].dt.month
df["Hauptkategorie"] = df["main_category_id"].map(object_id_to_name)
df["Unterkategorie"] = df["sub_category_id"].map(object_id_to_name)
# put time to resolution into bins
bins = [0, 1, 2, 4, 8, 24, 48, 72, 7 * 24, 14 * 24, 21 * 24]
df["time_to_resolution_bin"] = pd.cut(df["time_to_resolution_h"], bins=bins)
df["time_to_resolution_bin"] = df["time_to_resolution_bin"].apply(
    lambda x: f"{int(x.left)}–{int(x.right)}"
)
df.head(5)

In [ ]:
df.shape

In [ ]:
result = (
    df[df["month"] == 11][["created_string", "key"]]
    .groupby("created_string")
    .count()
    .reset_index()
)
print(result["key"].mean())
# plot using plotly
fig = px.bar(result, x="created_string", y="key", text="key")
# set width of plot
fig.update_layout(width=1000)
# add x label
fig.update_xaxes(title_text="Date")
# add y label
fig.update_yaxes(title_text="Count of tickets")
fig.show()

In [ ]:
df["request_type"].value_counts()

In [ ]:
# Plot request_type in bar chart
result = df[df["request_type"] != ""]["request_type"].value_counts().reset_index()
result["count"] = result["count"] / result["count"].sum()
fig = px.bar(result, x="request_type", y="count", text="count")
# set width of plot
# add share as labels inside of bars, as percentage
fig.update_traces(
    textposition="inside", insidetextanchor="middle", texttemplate="%{text:.1%}"
)
fig.update_layout(width=1000)
# add y axis label
fig.update_yaxes(title_text="Share of tickets")
fig.show()

## Open Ticket analysis

In [ ]:
dfopen = df[df["status_category"] != "Fertig"].copy()
todays_date = pd.Timestamp.now(tz="UTC")
dfopen["days_open"] = (todays_date - dfopen["created"]).dt.days
dfopen["weeks_open"] = -np.floor(dfopen["days_open"] / 7)
result = (
    dfopen[df["status_category"] == "In Arbeit"][
        ["weeks_open", "Hauptkategorie", "status_category"]
    ]
    .groupby(["weeks_open", "Hauptkategorie"])
    .count()
    .reset_index()
)

# plot: weeks open on x axis, count of tickets per category on y axis as stacked vertical bars
fig = px.bar(result, x="weeks_open", y="status_category", color="Hauptkategorie")
fig.update_layout(width=1000)
fig.show()

In [ ]:
dfopen = df[df["status_category"] != "Fertig"].copy()
todays_date = pd.Timestamp.now(tz="UTC")
dfopen["days_open"] = (todays_date - dfopen["created"]).dt.days
dfopen["weeks_open"] = -np.floor(dfopen["days_open"] / 7)
result = (
    dfopen[df["status_category"] == "In Arbeit"][["weeks_open", "status", "key"]]
    .groupby(["weeks_open", "status"])
    .count()
    .reset_index()
)

# plot: weeks open on x axis, count of tickets per category on y axis as stacked vertical bars
fig = px.bar(result, x="weeks_open", y="key", color="status")
fig.update_layout(width=1000)
fig.show()

In [ ]:
result = (
    df[df["status_category"] != "Fertig"][["status_category", "status", "key"]]
    .groupby(["status_category", "status"])
    .count()
    .reset_index()
    .sort_values("key", ascending=False)
)
# plot using plotly with status_category on x axis and status on y axis
fig = px.bar(result, x="status_category", y="key", color="status", text="status")
# add labels inside of bars
fig.update_traces(textposition="inside", insidetextanchor="middle")
# set width of plot
fig.update_layout(width=1000)
fig.update_layout(
    title="Status and Status Category for open tickets",
)
# make bars in shades of grey
fig.update_traces(marker_color=px.colors.qualitative.Plotly)
# add y axis label
fig.update_yaxes(title_text="Count of tickets")
# remove legend
fig.update_layout(showlegend=False, uniformtext_minsize=10)
fig.show()

In [ ]:
df[df["status_category"] == "In Arbeit"]["status"].value_counts()

In [ ]:
df[df["priority"].isin(["Hoch", "Sehr Hoch", "Rot"])][
    ["Hauptkategorie", "priority"]
].groupby("Hauptkategorie").count().reset_index()

## Time to resolution analysis

In [ ]:
# plot time to resolution bin counts using plotly
# sort by midpoint of intervals/bins
result = (
    df[df["currentstatus_name"] == "Fertig"][["time_to_resolution_bin", "key"]]
    .groupby("time_to_resolution_bin")
    .count()
    .reset_index()
)
result["key"] = result["key"] / result["key"].sum()
fig = px.bar(result, x="time_to_resolution_bin", y="key")
# add x axis label

fig.update_layout(
    title="Share of tickets by time to resolution",
)
fig.update_xaxes(title_text="Time to resolution in hours")
# add y axis label
fig.update_yaxes(title_text="Share of tickets")
# set width of plot
fig.update_layout(width=1000)
fig.show()

In [ ]:
# plot time to resolution bin counts using plotly
# sort by midpoint of intervals/bins
result = (
    df[df["currentstatus_name"] == "Fertig"][["bdays"]].value_counts().reset_index()
)
result["count"] = result["count"] / result["count"].sum()
fig = px.bar(result, x="bdays", y="count")
# add x axis label)
fig.update_layout(
    title="Share of tickets by time to resolution in business days",
)
fig.update_xaxes(title_text="Time to resolution in business days")
# add y axis label
fig.update_yaxes(title_text="Share of tickets")
# set x axis range
fig.update_xaxes(range=[-1, 16])
fig.update_yaxes(range=[0, 0.7])
# set x tick values
fig.update_xaxes(tickvals=list(range(0, 16)))
# set width of plot
fig.update_layout(width=1000)
fig.show()

In [ ]:
# Analyze average time to resolution for top 15 main categories
# compute average time and count by main category
top15 = (
    df[df["currentstatus_name"] == "Fertig"]["Hauptkategorie"]
    .value_counts()
    .head(14)
    .index
)
df_top15 = df[df["Hauptkategorie"].isin(top15)]

result = df[
    (df["currentstatus_name"] == "Fertig") & (df["Hauptkategorie"].isin(top15))
][["Hauptkategorie", "time_to_resolution_h"]]

# group by main category and compute median time to resolution as well as count
result = (
    result.groupby("Hauptkategorie")
    .agg(
        count=("Hauptkategorie", "size"),
        median_time_delta=("time_to_resolution_h", "median"),
    )
    .reset_index()
    .sort_values("count", ascending=False)
)

# Plot the main categories with a percentage bar in terms of count and include the cumulative count as a line
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bar trace
fig.add_trace(
    go.Bar(
        x=result["Hauptkategorie"],
        y=result["median_time_delta"],
        name="Median time to resolution in hours",
    ),
    secondary_y=False,
)

# Line trace (secondary y-axis)
fig.add_trace(
    go.Scatter(
        x=result["Hauptkategorie"],
        y=result["count"],
        mode="lines+markers",
        name="Count of tickets",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Median time to resolution in hours by main category (and count)",
    xaxis_title="Hauptkategorie",
    yaxis_title="Share of tickets",
)
fig.update_yaxes(title_text="Cumulative share", secondary_y=True)
# add x axis label in font size 8
fig.update_xaxes(title_text="Hauptkategorie", tickfont=dict(size=8))
fig.update_xaxes(tickangle=45)
# add y axis label
fig.update_yaxes(title_text="Median time to resolution in hours")
# set width of plot
fig.update_layout(width=1000)
fig.show()

In [ ]:
result = (
    df[df["Hauptkategorie"].isin(top15)][["Hauptkategorie", "resolution", "key"]]
    .groupby(["Hauptkategorie", "resolution"])
    .count()
    .reset_index()
)
# sort by overall count
result = result.sort_values("key", ascending=False)
# normalize per Hauptkategorie
result = result.groupby("Hauptkategorie").apply(
    lambda x: x.assign(Share=x["key"] / x["key"].sum())
)

fig = px.bar(result, x="Hauptkategorie", y="Share", color="resolution")
fig.update_layout(width=1000)
fig.show()

In [ ]:
result = (
    df[["Hauptkategorie", "resolution", "key"]]
    .groupby(["Hauptkategorie", "resolution"])
    .count()
    .reset_index()
)
result = result.rename(columns={"key": "Anzahl"})
# sort by overall count
result = result.sort_values("Anzahl", ascending=False)

fig = px.bar(result, x="Hauptkategorie", y="Anzahl", color="resolution")
fig.update_layout(width=1000)
fig.show()

In [ ]:
result

In [ ]:
result = (
    df[df["currentstatus_name"] == "Fertig"][["Hauptkategorie", "key"]]
    .groupby("Hauptkategorie")
    .count()
    .reset_index()
    .sort_values("key", ascending=False)
)
result["Cumulative share"] = result["key"].cumsum() / result["key"].sum()
result["Share"] = result["key"] / result["key"].sum()


# Plot the main categories with a percentage bar in terms of count and include the cumulative count as a line
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bar trace
fig.add_trace(
    go.Bar(
        x=result["Hauptkategorie"],
        y=result["Share"],
        name="Share",
    ),
    secondary_y=False,
)

# Line trace (secondary y-axis)
fig.add_trace(
    go.Scatter(
        x=result["Hauptkategorie"],
        y=result["Cumulative share"],
        mode="lines+markers",
        name="Cumulative share",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Distribution of tickets by main category",
    xaxis_title="Hauptkategorie",
    yaxis_title="Share of tickets",
)
fig.update_yaxes(title_text="Cumulative share", secondary_y=True)
# add x axis label in font size 8
fig.update_xaxes(title_text="Hauptkategorie", tickfont=dict(size=9))
fig.update_xaxes(tickangle=45)
# add y axis label
fig.update_yaxes(title_text="Share of tickets")
# set width of plot
fig.update_layout(width=1200)
fig.update_layout(height=500)
fig.show()

## Analyzing topics

In [ ]:
print("Top 15 main categories:", list(top15))

### Get tickets for September-November per category

In [ ]:
dff_oct2025 = df[(df["month"].isin([9, 10, 11])) & (df["Hauptkategorie"].isin(top15))]

In [ ]:
dff_oct2025.shape

In [ ]:
summaries = []
for cat in top15:
    descriptions = dff_oct2025[dff_oct2025["Hauptkategorie"] == cat]["description"]
    titles = dff_oct2025[dff_oct2025["Hauptkategorie"] == cat]["summary"]
    descriptions = [f"{t}\n{d}" for t, d in zip(titles, descriptions)]
    summary = get_summary(descriptions)
    summaries.append(f"## {cat}\n\n{summary}")

summaries_markdown = "\n\n".join(summaries)
# save summaries to file
with open("summaries.md", "w") as f:
    f.write(summaries_markdown)

In [ ]:
dff_oct2025_sameday = dff_oct2025[dff_oct2025["bdays"] == 0]
summaries = []
for cat in top15:
    descriptions = dff_oct2025_sameday[dff_oct2025_sameday["Hauptkategorie"] == cat][
        "description"
    ]
    titles = dff_oct2025_sameday[dff_oct2025_sameday["Hauptkategorie"] == cat][
        "summary"
    ]
    descriptions = [f"{t}\n{d}" for t, d in zip(titles, descriptions)]
    summary = get_summary(descriptions)
    summaries.append(f"## {cat}\n\n{summary}")

summaries_markdown = "\n\n".join(summaries)
# save summaries to file
with open("summaries_sameday.md", "w") as f:
    f.write(summaries_markdown)

In [ ]:
dff_oct2025_emails = dff_oct2025[dff_oct2025["request_type"] == "Anfrage per E-Mail"]
summaries = []
for cat in top15:
    descriptions = dff_oct2025_emails[dff_oct2025_emails["Hauptkategorie"] == cat][
        "description"
    ]
    titles = dff_oct2025_emails[dff_oct2025_emails["Hauptkategorie"] == cat]["summary"]
    descriptions = [f"{t}\n{d}" for t, d in zip(titles, descriptions)]
    summary = get_summary(descriptions)
    summaries.append(f"## {cat}\n\n{summary}")

summaries_markdown = "\n\n".join(summaries)
# save summaries to file
with open("summaries_emails.md", "w") as f:
    f.write(summaries_markdown)

In [ ]:
dff_oct2025_emails

## Focus on E-Mail tickets

In [ ]:
dfem = df[df["request_type"] == "Anfrage per E-Mail"]

In [ ]:
dfem.shape

## Clone relationships

In [ ]:
def get_clone_relations(issue):
    cloned_from = None
    clones = []

    for link in issue.fields.issuelinks:
        t = link.type

        # this issue was cloned FROM another
        if hasattr(link, "outwardIssue") and t.name == "Cloners":
            cloned_from = link.outwardIssue.key

        # this issue was cloned BY another
        if hasattr(link, "inwardIssue") and t.name == "Cloners":
            clones.append(link.inwardIssue.key)

    return cloned_from, clones

In [ ]:
issues2 = issues[:50]
for issue in issues2:
    issue = jira.issue(issue.key)
    cloned_from, clones = get_clone_relations(issue)

    if cloned_from is not None and cloned_from not in clones:
        print(issue.key, issue.fields.summary, "Cloned from:", cloned_from)
    if len(clones) > 0:
        print(issue.key, issue.fields.summary, "Clones:", clones)